# Practice with PyTorch Forecasting, HuggingFace Datasets, and MAPIE

This notebook contains code for the experimentation with the pytorch-forecasting, datasets, and mapie Python libraries to help with the development of the project. It will consist of importing the data with datasets, constructing a temporal fusion transformer (TFT), and using MAPIE to create conformal prediction intervals. No imputation.

In [7]:
# Imports
import datetime as dt
import warnings
from datasets import load_dataset

warnings.simplefilter("ignore")

In [8]:
# import the dataset
dataset = load_dataset(
    "Dingdong-Inc/FreshRetailNet-50K",
    cache_dir="/home/gil/projects/uncertainty_aware_planner/data",
)
dataset.cleanup_cache_files()

{'train': 0, 'eval': 0}

In [9]:
# examining the dataset through a streaming interface given the size

train_set = dataset['train'].to_iterable_dataset()
train_set_slice = train_set.take(5)
for ex in train_set_slice:
    print(ex)

{'city_id': 0, 'store_id': 0, 'management_group_id': 0, 'first_category_id': 5, 'second_category_id': 6, 'third_category_id': 65, 'product_id': 38, 'dt': '2024-03-28', 'sale_amount': 0.1, 'hours_sale': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'stock_hour6_22_cnt': 0, 'hours_stock_status': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'discount': 1.0, 'holiday_flag': 0, 'activity_flag': 0, 'precpt': 1.6999, 'avg_temperature': 15.48, 'avg_humidity': 73.54, 'avg_wind_level': 1.97}
{'city_id': 0, 'store_id': 0, 'management_group_id': 0, 'first_category_id': 5, 'second_category_id': 6, 'third_category_id': 65, 'product_id': 38, 'dt': '2024-03-29', 'sale_amount': 0.1, 'hours_sale': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'stock_hour6_22_cnt': 1, 'hours_stock_status': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [10]:
# map processing of datetime information
hours = [dt.time(hour=i) for i in range(24)]

def add_datetimes(example):
    example_date = dt.date.fromisoformat(example["dt"])
    example["hours"] = hours
    example["datetime"] = [dt.datetime.combine(date=example_date, time=t) for t in hours]
    return example

train_set = train_set.map(add_datetimes)
train_set_slice = train_set.take(5)
for ex in train_set_slice:
    print(ex)

{'city_id': 0, 'store_id': 0, 'management_group_id': 0, 'first_category_id': 5, 'second_category_id': 6, 'third_category_id': 65, 'product_id': 38, 'dt': '2024-03-28', 'sale_amount': 0.1, 'hours_sale': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'stock_hour6_22_cnt': 0, 'hours_stock_status': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'discount': 1.0, 'holiday_flag': 0, 'activity_flag': 0, 'precpt': 1.6999, 'avg_temperature': 15.48, 'avg_humidity': 73.54, 'avg_wind_level': 1.97, 'hours': [datetime.time(0, 0), datetime.time(1, 0), datetime.time(2, 0), datetime.time(3, 0), datetime.time(4, 0), datetime.time(5, 0), datetime.time(6, 0), datetime.time(7, 0), datetime.time(8, 0), datetime.time(9, 0), datetime.time(10, 0), datetime.time(11, 0), datetime.time(12, 0), datetime.time(13, 0), datetime.time(14, 0), datetime.time(15, 0), datetime.time(16, 0), datetime.time(17, 0), datetime.tim

In [11]:
# map explosion of each day into its component hours and return back to normal formatting
train_set = train_set.with_format('pandas')
train_set = train_set.map(lambda df: df.explode(['hours_sale', 'hours_stock_status', 'hours', 'datetime']), batched=True)
train_set = train_set.map(lambda df: df.reset_index())
train_set = train_set.with_format(None)
train_set_slice = train_set.take(40)
for ex in train_set_slice:
    print(ex)

{'index': 0, 'city_id': 0, 'store_id': 0, 'management_group_id': 0, 'first_category_id': 5, 'second_category_id': 6, 'third_category_id': 65, 'product_id': 38, 'dt': '2024-03-28', 'sale_amount': 0.1, 'hours_sale': 0.0, 'stock_hour6_22_cnt': 0, 'hours_stock_status': 0, 'discount': 1.0, 'holiday_flag': 0, 'activity_flag': 0, 'precpt': 1.6999, 'avg_temperature': 15.48, 'avg_humidity': 73.54, 'avg_wind_level': 1.97, 'hours': datetime.time(0, 0), 'datetime': datetime.datetime(2024, 3, 28, 0, 0)}
{'index': 0, 'city_id': 0, 'store_id': 0, 'management_group_id': 0, 'first_category_id': 5, 'second_category_id': 6, 'third_category_id': 65, 'product_id': 38, 'dt': '2024-03-28', 'sale_amount': 0.1, 'hours_sale': 0.0, 'stock_hour6_22_cnt': 0, 'hours_stock_status': 0, 'discount': 1.0, 'holiday_flag': 0, 'activity_flag': 0, 'precpt': 1.6999, 'avg_temperature': 15.48, 'avg_humidity': 73.54, 'avg_wind_level': 1.97, 'hours': datetime.time(1, 0), 'datetime': datetime.datetime(2024, 3, 28, 1, 0)}
{'index'

In [12]:
# getting the pre-requisite time index for forecasting
beginning_datetime = dt.datetime(2024, 3, 28, 0, 0)

def add_time_idx(example):
    time_from_beginning = (example['datetime'] - beginning_datetime)
    seconds_from_beginning = time_from_beginning.days * 86400 + time_from_beginning.seconds
    example['time_idx'] = int(seconds_from_beginning / 3600)
    return example
    
train_set = train_set.map(add_time_idx)
train_set_slice = train_set.take(40)
for ex in train_set_slice:
    print(ex)

{'index': 0, 'city_id': 0, 'store_id': 0, 'management_group_id': 0, 'first_category_id': 5, 'second_category_id': 6, 'third_category_id': 65, 'product_id': 38, 'dt': '2024-03-28', 'sale_amount': 0.1, 'hours_sale': 0.0, 'stock_hour6_22_cnt': 0, 'hours_stock_status': 0, 'discount': 1.0, 'holiday_flag': 0, 'activity_flag': 0, 'precpt': 1.6999, 'avg_temperature': 15.48, 'avg_humidity': 73.54, 'avg_wind_level': 1.97, 'hours': datetime.time(0, 0), 'datetime': datetime.datetime(2024, 3, 28, 0, 0), 'time_idx': 0}
{'index': 0, 'city_id': 0, 'store_id': 0, 'management_group_id': 0, 'first_category_id': 5, 'second_category_id': 6, 'third_category_id': 65, 'product_id': 38, 'dt': '2024-03-28', 'sale_amount': 0.1, 'hours_sale': 0.0, 'stock_hour6_22_cnt': 0, 'hours_stock_status': 0, 'discount': 1.0, 'holiday_flag': 0, 'activity_flag': 0, 'precpt': 1.6999, 'avg_temperature': 15.48, 'avg_humidity': 73.54, 'avg_wind_level': 1.97, 'hours': datetime.time(1, 0), 'datetime': datetime.datetime(2024, 3, 28, 